# Bayesian Optimization for Black-Box Functions (Submission 2)

This notebook implements a Bayesian Optimization framework using Gaussian Process Regression (GPR) to propose the next optimal query points for 8 distinct black-box functions. 

### Key Methodology Features:
* **Matern 5/2 Kernel:** Selected to allow for realistic irregularities in the objective functions without the strict smoothness assumptions of an RBF kernel.
* **Robust Scale Handling:** `normalize_y=True` is enabled within the GPR to robustly handle outputs spanning across massive ranges (e.g., $10^{-50}$ to $3486.0$).
* **Acquisition Function:** Expected Improvement (EI) with an exploration parameter ($\xi = 0.01$) to prevent greedy local convergence.
* **Global Acquisition Optimization:** Employs L-BFGS-B with 30 random restarts to reliably navigate the non-convex acquisition surface.

In [ ]:
import numpy as np
import os
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm

# Ensure inline plotting if visualizations are added later
%matplotlib inline 

print("Libraries successfully imported.")

## Defining the Bayesian Optimizer Class

The `BayesianOptimizer` class encapsulates data ingestion, surrogate model fitting, acquisition function evaluation, and the multi-start gradient-based optimization phase.

In [ ]:
class BayesianOptimizer:
    def __init__(self, is_noisy=False):
        """
        Initializes the optimizer.
        
        Parameters:
        -----------
        is_noisy : bool
            If True, uses a higher noise variance (alpha) to account for noisy likelihoods.
        """
        # We use a higher alpha (1e-2) for Function 2 to handle its "noisy likelihood"
        self.alpha = 1e-2 if is_noisy else 1e-6
        self.model = None
        self.X = None
        self.Y = None
        self.dim = None

    def load_and_fit(self, func_dir):
        """
        Loads the combined initial and historic dataset, builds the GP kernel,
        and fits the Gaussian Process Regressor.
        """
        # Load dataset (including cumulative historic results up to current round)
        X = np.load(os.path.join(func_dir, "initial_inputs.npy"))
        Y = np.load(os.path.join(func_dir, "initial_outputs.npy"))
        self.X, self.Y, self.dim = X, Y, X.shape[1]
        
        # Kernel configuration: Matern 5/2 handles localized irregularities well
        kernel = ConstantKernel(1.0) * Matern(
            length_scale=np.ones(self.dim), 
            length_scale_bounds=(0.01, 1.0), 
            nu=2.5
        )
        
        # normalize_y=True scales targets to mean 0, variance 1 to handle diverse ranges
        self.model = GaussianProcessRegressor(
            kernel=kernel, 
            alpha=self.alpha, 
            normalize_y=True,
            n_restarts_optimizer=25
        )
        self.model.fit(self.X, self.Y)

    def expected_improvement(self, x, xi=0.01):
        """
        Computes the Expected Improvement (EI) at point x.
        
        Parameters:
        -----------
        xi : float
            Exploration-exploitation trade-off parameter. Higher values favor exploration.
        """
        x = x.reshape(-1, self.dim)
        mu, sigma = self.model.predict(x, return_std=True)
        mu_sample_opt = np.max(self.Y)

        with np.errstate(divide='warn'):
            imp = mu - mu_sample_opt - xi
            Z = imp / sigma
            ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
            ei[sigma <= 0.0] = 0.0
        return ei

    def propose_next_point(self):
        """
        Optimizes the acquisition function surface using 30 random restarts
        of the L-BFGS-B local optimizer to find the true global maximum.
        """
        best_x = None
        max_ei = -1
        
        # Restarts ensure we overcome the non-convexity of the acquisition surface
        for _ in range(30):
            x0 = np.random.uniform(0.0, 0.999999, self.dim)
            res = minimize(
                lambda x: -self.expected_improvement(x),
                x0,
                bounds=[(0.0, 0.999999)] * self.dim,
                method='L-BFGS-B'
            )
            if -res.fun > max_ei:
                max_ei = -res.fun
                best_x = res.x
        return best_x

## Sequential Optimization Loop (Functions 1–8)

We iterate through each target function directory, dynamically adjust settings for noisy objectives (Function 2), fit the GPR surrogate model, and output the targeted query parameters to our submission file.

In [ ]:
output_file = "submission_2_results.txt"

print(f"Calculating Round 2 Queries...")
print("-" * 40)

# Dictionary to collect results for notebook summary display
notebook_results = {}

with open(output_file, "w") as f:
    for i in range(1, 9):
        func_dir = f"function_{i}"
        
        # Check if directories exist before running to avoid notebook errors
        if not os.path.exists(func_dir):
            print(f"Warning: Directory '{func_dir}' not found. Skipping.")
            continue
            
        # Instantiate and run optimizer
        optimizer = BayesianOptimizer(is_noisy=(i == 2))
        optimizer.load_and_fit(func_dir)
        
        next_point = optimizer.propose_next_point()
        formatted_point = "-".join([f"{val:.6f}" for val in next_point])
        
        result_line = f"Function {i}: {formatted_point}"
        f.write(result_line + "\n")
        
        # Cache results for displaying below
        notebook_results[f"Function {i}"] = formatted_point
        print(result_line)

print("-" * 40)
print(f"Submission 2 successfully generated and saved to: {output_file}")

### Summary Table of Proposed Queries

In [ ]:
# Neatly print out the results summary inside the notebook tracking
print(f"| {'Function Target':15} | {'Next Proposed Query Coordinates':50} |")
print(f"| {'-'*15} | {'-'*50} |")
for func, points in notebook_results.items():
    print(f"| {func:15} | {points:50} |")